# Run everything

Executes the project notebooks **in order** (01 to 09, then 11 and 10), saves each notebook's outputs, and stops at the first error. Notebook 00 (download) runs only if `data/raw/online_retail_II.xlsx` is missing.

**Run it:** open this notebook from the `notebooks/` folder and choose *Run All* (about 6 minutes; the Excel load in 01 is the slowest step), or from a terminal in the `notebooks/` folder:

```
..\.venv\Scripts\python -m nbconvert --to notebook --execute --inplace run_all.ipynb
```

**Run only some notebooks:** set the environment variable `RUN_ONLY` to a comma-separated list of notebook names, for example `RUN_ONLY=04_rfm_segmentation,05_churn_and_hypothesis_tests`. Notebook order matters: run 08 before 09, and the analysis notebooks before 08.

Notebook 09 writes only `retail_dashboard_generated.*`, so this never overwrites the Tableau workbook that was edited by hand.

In [ ]:
import os
import time
from pathlib import Path

import nbformat
import pandas as pd
from nbclient import NotebookClient

NB_DIR = Path.cwd()  # run from the notebooks/ folder
RAW_FILE = NB_DIR.parent / "data" / "raw" / "online_retail_II.xlsx"
ORDER = [
    "00_download_data", "01_data_cleaning", "02_eda", "03_cohort_retention", "04_rfm_segmentation",
    "05_churn_and_hypothesis_tests", "06_sql_analysis", "07_recommendations", "08_build_dashboard", "09_build_tableau_workbook",
    "11_robustness_checks", "10_documentation_tables",  # 11 runs before 10: the documentation tables read its results
]
ONLY = [n.strip() for n in os.environ.get("RUN_ONLY", "").split(",") if n.strip()]
TIMEOUT = 1800  # seconds allowed per notebook

unknown = [n for n in ONLY if n not in ORDER]
assert not unknown, f"unknown notebook names in RUN_ONLY: {unknown}"
to_run = [n for n in ORDER if n in ONLY] if ONLY else [n for n in ORDER if not (n == "00_download_data" and RAW_FILE.exists())]
to_run

In [ ]:
results = []
for name in to_run:
    path = NB_DIR / f"{name}.ipynb"
    nb = nbformat.read(path, as_version=4)
    started = time.time()
    client = NotebookClient(nb, timeout=TIMEOUT, kernel_name="python3", resources={"metadata": {"path": str(NB_DIR)}})
    try:
        client.execute()
    except Exception as err:
        results.append((name, f"FAILED ({type(err).__name__})", round(time.time() - started)))
        print(pd.DataFrame(results, columns=["notebook", "status", "seconds"]).to_string(index=False))
        raise
    nbformat.write(nb, path)
    results.append((name, "ok", round(time.time() - started)))
    print(f"{name}: ok in {results[-1][2]}s")


In [ ]:
pd.DataFrame(results, columns=["notebook", "status", "seconds"])